In [ ]:
import nltk

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('movie_reviews')

In [ ]:
from nltk.corpus import movie_reviews

categories = movie_reviews.categories()

sentences = []
targets = []

for file_id in movie_reviews.fileids():
    words = movie_reviews.words(file_id)
    sentences.append(' '.join(words))
    targets.append(categories.index(movie_reviews.categories(file_id)[0]))

print(words[:5])
print(sentences[:5])
print(targets[:5])

# Exercice 1

Use the ```text_tokenized```  function from nltk to tokenize the sentences and store everything in a ```text_tokenized``` list.

In [ ]:
text_tokenized = [nltk.word_tokenize(text) for text in sentences]

In [ ]:
for i in range(5):
    print(text_tokenized[i])

Apply Punctuation and special characters removal

In [ ]:
import string

print(string.punctuation)

text_no_punctuation = []

In [ ]:
for sentence in text_tokenized:
    sentence_new = []
    for word in sentence:
        for character in string.punctuation:
            word = word.replace(character, '')
        if word != '':
            sentence_new.append(word)
    text_no_punctuation.append(sentence_new)

In [ ]:
text_no_punctuation[0]

Using nltk.corpus.stopwords, remove all stopwords from the sentences

In [ ]:
from nltk.corpus import stopwords

text_no_stop_words = []


In [ ]:
stoplist = stopwords.words('english')

for sentence in text_no_punctuation:
    text_no_stop_words.append([word for word in sentence if word not in stoplist])

In [ ]:
print('Number of words before removing stop words: %d' %len(text_no_punctuation[0]))
print('Number of words after removing stop words: %d' %len(text_no_stop_words[0]))

print(len(text_no_stop_words[0]))

Apply the PorterStemmer Algorithm from ```nltk.stem.porter.PorterStemmer```to keep only stems from the sentences.

In [ ]:
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()
text_stemmed = []

In [ ]:
for sentence in text_no_stop_words:
    text_stemmed.append([porter.stem(word) for word in sentence])

In [ ]:
text_stemmed[0]

##  Bag of words

Create a dataset with the sentences and fit a Bag of Words model using the following imports.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [ ]:
clean_sentences = [' '.join(sentence) for sentence in text_stemmed]

In [ ]:
sentence_train, sentence_test, y_train, y_test = train_test_split(clean_sentences, targets, shuffle=True, test_size=0.1)

In [ ]:
count_vectorizer = CountVectorizer()
X_train = count_vectorizer.fit_transform(sentence_train)
X_test = count_vectorizer.transform(sentence_test)

Build a Random Forest Classifier to predict the sentiment of the movie reviews. Analyze the performance of the classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rm_model = RandomForestClassifier()

rm_model.fit(X_train, y_train)

In [ ]:
preds_train = rm_model.predict(X_train)
print('Accuracy for training set is : %.4f' % (sum(y_train == preds_train)/len(preds_train)))

preds_test = rm_model.predict(X_test)
print('Accuracy for training set is : %.4f' % (sum(y_test == preds_test)/len(preds_test)))

## Exercice 10 - glove (vector representation of words)

Download the embeddings from http://nlp.stanford.edu/data/glove.6B.zip and load them.

In [ ]:
import numpy as np
from scipy import spatial

embeddings = {}
with open("glove.6B.50d.txt", 'r', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], 'float32')
        embeddings[word] = vector

Create a function using the euclidian distance to find the closest embeddings from a certain vector representation.

In [ ]:
def find_closer(vector, embeddings):
    return(sorted(embeddings.keys(), key=lambda word: spatial.distance.euclidean( embeddings[word], vector)))

Find the words that have an embedding that is close to the embedding of the word 'king'.

In [ ]:
find_closer(embeddings['king'] ,embeddings)

Find the words that have an embedding that is close to the embedding of the word 'sad'.

In [ ]:
find_closer(embeddings['sad'], embeddings)

Create a vector representing :
- the embedding of 'king' - the embedding of 'man' + the embedding of 'woman'

Find the words that have an embedding that is close to the obtained embedding.

In [ ]:
vector = embeddings['king'] - embeddings['man'] + embeddings['woman']
find_closer(vector, embeddings)

## Visualize

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import pandas as pd
from sklearn.manifold import TSNE

def plot_words(word1, word2, embeddings, n=100):
    words_1 = find_closer(embeddings[word1],embeddings)
    words_2 = find_closer(embeddings[word2], embeddings)
    words1 = words_1[:n]
    words2 = words_2[:n]
    
    words1.extend(words2)

    vectors = []
    for word in words1:
        vectors.append(embeddings[word])

    tsne = TSNE(n_components=2, random_state=0)
    tsne_result = tsne.fit_transform(vectors)
    tsne_df = pd.DataFrame({'X': tsne_result[:,0], 'Y': tsne_result[:,1], 'word_similarity': [word1]*n+[word2]*n})

    sns.lmplot('X', 'Y', tsne_df, hue='word_similarity', fit_reg=False)
    plt.annotate(word1, (tsne_result[words1.index(word1), 0], tsne_result[words1.index(word1), 1]))
    plt.annotate(word2, (tsne_result[words1.index(word2), 0], tsne_result[words1.index(word2), 1]))

    plt.show()

In [ ]:
plot_words('computer', 'doctor', embeddings)

In [ ]:
plot_words('history', 'mathematics', embeddings)

## Exercice 11 - Try your visualization